# Обогащение данных

В этом ноутбуке покажем, как мы слили данные воедино

На данный момент мы имеем следующие наборы данных:

In [4]:
!tree datasets

datasets
├── Arpita-deb-netflix-movies-and-tv-shows
│   ├── README.md
│   ├── country.csv
│   ├── credits_cleaned.csv
│   ├── iso.csv
│   └── titles_cleaned.csv
├── Netflix
│   ├── NetFlix.csv
│   └── data_description.txt
├── Netflix Movies and TV Shows
│   ├── data_description.txt
│   └── netflix_titles.csv
├── Netflix Movies and TV shows till 2025
│   ├── data_description.txt
│   ├── netflix_movies_detailed_up_to_2025.csv
│   └── netflix_tv_shows_detailed_up_to_2025.csv
├── netflix_merged.tsv
└── original
    ├── NetflixShows.csv
    └── data_description.txt

6 directories, 15 files


Обратим внимение на то, что по заданию мы работаем с платформой **Netflix**, поэтому исходные датасеты были собраны именно на этой платформе

Поскольку датасетов немного (всего 5 штук), мы не будем писать умный скрипт для их объединения, а сделаем это руками, опираясь на описание каждого датасета - в нашем случае это быстрее и проще.

Канонические названия полей:  
* `title` - тайтл  
* `title_normalized` - нормализованный тайтл  
* `type` - TV Show или Movie  
* `release_year` - год релиза  
* `rating` -  американская система возрастных рейтингов   
* `rating_level` - текстовое описание возрастных рейтингов  
* `user_rating_score` - пользовательская оценка  
* `user_rating_size` - размер выборки для пользовательской оценки  
* `director` - режиссер  
* `country` - страна  
* `cast` - актерский состав  
* `genres` - жанры  
* `language` - язык  
* `description` - краткое описание  
* `budget` - бюджет  
* `revenue` - сборы  
* `duration` - продолжительность  
* `popularity` - популярность  
* `listed_in` - представлен в категориях
* `source_dataset` - файл-источник
* `source_row_id` - id в источнике

In [2]:
import pandas as pd
import numpy as np

### Начнем с предоставленного датасета

#### Функция для нормализации тайтла

In [ ]:
import re
import unicodedata

def strip_accents(text):
    text = unicodedata.normalize('NFKD', text)
    return ''.join(ch for ch in text if not unicodedata.combining(ch))

def normalize_title(title):
    if pd.isna(title):
        return pd.NA

    title = str(title).strip().lower()

    title = strip_accents(title)

    title = (title
             .replace('’', "'")
             .replace('‘', "'")
             .replace('`', "'")
             .replace('“', '"')
             .replace('”', '"')
             .replace('–', '-')
             .replace('—', '-')
             .replace('&', ' and '))

    title = re.sub(r'[:;,!?./]+', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()

    return title

In [4]:
NetflixShows_original = pd.read_csv('datasets/original/NetflixShows.csv').drop('Unnamed: 0', axis=1)

NetflixShows_original['title_normalized'] = NetflixShows_original['title'].apply(normalize_title)
NetflixShows_original['source_dataset'] = 'original'
NetflixShows_original['source_row_id'] = np.arange(1, len(NetflixShows_original) + 1)
NetflixShows_original['type'] = 'TV Show'

NetflixShows_original.rename(columns={
    'id' : 'source_row_id',
    'release year' : 'release_year',
    'user rating score' : 'user_rating_score',
    'user rating size' : 'user_rating_size',

}, inplace=True)

NetflixShows_original.drop_duplicates(subset=['title', 'release_year'], inplace=True)
NetflixShows_original.drop(['ratingLevel', 'ratingDescription'], axis=1, inplace=True)
NetflixShows_original

,title,rating,release_year,user_rating_score,user_rating_size,title_normalized,source_dataset,source_row_id,type
0,White Chicks,PG-13,2004,82.0,80,white chicks,original,1,TV Show
1,Lucky Number Slevin,R,2006,NaN,82,lucky number slevin,original,2,TV Show
2,Grey's Anatomy,TV-14,2016,98.0,80,grey's anatomy,original,3,TV Show
3,Prison Break,TV-14,2008,98.0,80,prison break,original,4,TV Show
4,How I Met Your Mother,TV-PG,2014,94.0,80,how i met your mother,original,5,TV Show
...,...,...,...,...,...,...,...,...,...
989,Russell Madness,PG,2015,NaN,82,russell madness,original,990,TV Show
993,Wiener Dog Internationals,G,2015,NaN,82,wiener dog internationals,original,994,TV Show
994,Pup Star,G,2016,NaN,82,pup star,original,995,TV Show
997,Precious Puppies,TV-G,2003,NaN,82,precious puppies,original,998,TV Show


### Netflix Movies and TV shows till 2025

In [5]:
netflix_movies_detailed_up_to_2025 = pd.read_csv('datasets/Netflix Movies and TV shows till 2025/netflix_movies_detailed_up_to_2025.csv')
netflix_movies_detailed_up_to_2025

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,1440286,Movie,Festival de Viña del Mar 2025: Ha*Ash,NaN,Ha*ash,Chile,2025-02-24,2025,0.000,NaN,Music,es,NaN,4.931,0,0.000,0,0
15996,1271724,Movie,Man and Woman,Vladimir Kott,"Anna Kotova, Stepan Devonin, Pavel Derevyanko,...",Russia,2025-03-13,2025,0.000,NaN,Drama,ru,Poignant stories about men and women who have ...,4.930,0,0.000,0,0
15997,1426364,Movie,Night of the Dead Sorority Babes,"Angel Nichole Bradford, Steve Hermann","Jessa Flux, Lynn Lowry, Angel Nichole Bradford...",NaN,2025-01-28,2025,1.000,NaN,Horror,en,Two villainous entities initiate gorgeous soro...,4.922,1,1.000,0,0
15998,1411248,Movie,A Dunces Burden,Daniel Kowal,"Riley G, Mitchel Corrado",NaN,2025-03-10,2025,0.000,NaN,NaN,en,A Dunces Burden,4.921,0,0.000,0,0


In [6]:
netflix_movies_detailed_up_to_2025['title_normalized'] = netflix_movies_detailed_up_to_2025['title'].apply(normalize_title)
netflix_movies_detailed_up_to_2025['date_added'] = pd.to_datetime(netflix_movies_detailed_up_to_2025['date_added'])
netflix_movies_detailed_up_to_2025['source_dataset'] = 'netflix_movies_detailed_up_to_2025'
netflix_movies_detailed_up_to_2025.drop('rating', axis=1, inplace=True) # признак vote_average дублирует
netflix_movies_detailed_up_to_2025.rename(
    columns={
        'show_id' : 'source_row_id',
        'vote_count' : 'user_rating_size',
        'vote_average' : 'user_rating_score'
    }, inplace=True)
netflix_movies_detailed_up_to_2025['user_rating_score'] = netflix_movies_detailed_up_to_2025['user_rating_score'] * 10
netflix_movies_detailed_up_to_2025

,source_row_id,type,title,director,cast,country,date_added,release_year,duration,genres,language,description,popularity,user_rating_size,user_rating_score,budget,revenue,title_normalized,source_dataset
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,63.80,165000000,752600867,shrek forever after,netflix_movies_detailed_up_to_2025
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,83.69,160000000,839030630,inception,netflix_movies_detailed_up_to_2025
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,77.44,250000000,954305868,harry potter and the deathly hallows part 1,netflix_movies_detailed_up_to_2025
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,76.00,260000000,592461732,tangled,netflix_movies_detailed_up_to_2025
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,78.00,165000000,494879471,how to train your dragon,netflix_movies_detailed_up_to_2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,1440286,Movie,Festival de Viña del Mar 2025: Ha*Ash,NaN,Ha*ash,Chile,2025-02-24,2025,NaN,Music,es,NaN,4.931,0,0.00,0,0,festival de vina del mar 2025 ha*ash,netflix_movies_detailed_up_to_2025
15996,1271724,Movie,Man and Woman,Vladimir Kott,"Anna Kotova, Stepan Devonin, Pavel Derevyanko,...",Russia,2025-03-13,2025,NaN,Drama,ru,Poignant stories about men and women who have ...,4.930,0,0.00,0,0,man and woman,netflix_movies_detailed_up_to_2025
15997,1426364,Movie,Night of the Dead Sorority Babes,"Angel Nichole Bradford, Steve Hermann","Jessa Flux, Lynn Lowry, Angel Nichole Bradford...",NaN,2025-01-28,2025,NaN,Horror,en,Two villainous entities initiate gorgeous soro...,4.922,1,10.00,0,0,night of the dead sorority babes,netflix_movies_detailed_up_to_2025
15998,1411248,Movie,A Dunces Burden,Daniel Kowal,"Riley G, Mitchel Corrado",NaN,2025-03-10,2025,NaN,NaN,en,A Dunces Burden,4.921,0,0.00,0,0,a dunces burden,netflix_movies_detailed_up_to_2025


In [7]:
netflix_tv_shows_detailed_up_to_2025 = pd.read_csv('datasets/Netflix Movies and TV shows till 2025/netflix_tv_shows_detailed_up_to_2025.csv')
netflix_tv_shows_detailed_up_to_2025

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha...",South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members a...,1929.898,187,8.241
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by C...,1670.580,229,7.035
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooki...,1317.092,6,5.600
3,75685,TV Show,Prostřeno!,NaN,"Václav Vydra, Jana Boušková",Czech Republic,2010-03-01,2010,6.500,1 Seasons,Reality,cs,The knives (and forks) are out as a group of s...,1095.776,6,6.500
4,33847,TV Show,The Talk,NaN,"Amanda Kloots, Jerry O'Connell, Akbar Gbaja-Bi...","United States of America, Ireland",2010-10-18,2010,3.400,1 Seasons,Talk,en,A panel of well-known news and entertainment p...,712.070,12,3.400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,284892,TV Show,Sammelanam,NaN,NaN,NaN,2025-02-20,2025,0.000,1 Seasons,"Comedy, Drama",te,Sammelanam is a gripping Telugu web series tha...,3.236,0,0.000
15996,277665,TV Show,Anne Shirley,NaN,"Honoka Inoue, Yasunori Matsumoto, Aya Nakamura...",Japan,2025-04-05,2025,0.000,1 Seasons,"Animation, Drama, Family",ja,It's depict Anne's growth from a girl to a wom...,3.558,0,0.000
15997,284972,TV Show,Le onde del passato,Giulio Manfredonia,"Anna Valle, Giorgio Marchesi, Irene Ferri, Fau...",NaN,2025-02-19,2025,10.000,1 Seasons,Drama,it,NaN,2.913,1,10.000
15998,284613,TV Show,AI히치하이커,NaN,NaN,NaN,2025-02-16,2025,0.000,1 Seasons,NaN,ko,NaN,2.787,0,0.000


In [8]:
netflix_tv_shows_detailed_up_to_2025['title_normalized'] = netflix_tv_shows_detailed_up_to_2025['title'].apply(normalize_title)
netflix_tv_shows_detailed_up_to_2025['date_added'] = pd.to_datetime(netflix_tv_shows_detailed_up_to_2025['date_added'])
netflix_tv_shows_detailed_up_to_2025['source_dataset'] = 'netflix_tv_shows_detailed_up_to_2025'
netflix_tv_shows_detailed_up_to_2025.drop('rating', axis=1, inplace=True) # признак vote_average дублирует
netflix_tv_shows_detailed_up_to_2025.rename(
    columns={
        'show_id' : 'source_row_id',
        'vote_count' : 'user_rating_size',
        'vote_average' : 'user_rating_score'
    }, inplace=True)
netflix_tv_shows_detailed_up_to_2025['user_rating_score'] = netflix_tv_shows_detailed_up_to_2025['user_rating_score'] * 10
netflix_tv_shows_detailed_up_to_2025

,source_row_id,type,title,director,cast,country,date_added,release_year,duration,genres,language,description,popularity,user_rating_size,user_rating_score,title_normalized,source_dataset
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha...",South Korea,2010-07-11,2010,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members a...,1929.898,187,82.41,running man,netflix_tv_shows_detailed_up_to_2025
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by C...,1670.580,229,70.35,conan,netflix_tv_shows_detailed_up_to_2025
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooki...,1317.092,6,56.00,masterchef greece,netflix_tv_shows_detailed_up_to_2025
3,75685,TV Show,Prostřeno!,NaN,"Václav Vydra, Jana Boušková",Czech Republic,2010-03-01,2010,1 Seasons,Reality,cs,The knives (and forks) are out as a group of s...,1095.776,6,65.00,prostreno,netflix_tv_shows_detailed_up_to_2025
4,33847,TV Show,The Talk,NaN,"Amanda Kloots, Jerry O'Connell, Akbar Gbaja-Bi...","United States of America, Ireland",2010-10-18,2010,1 Seasons,Talk,en,A panel of well-known news and entertainment p...,712.070,12,34.00,the talk,netflix_tv_shows_detailed_up_to_2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,284892,TV Show,Sammelanam,NaN,NaN,NaN,2025-02-20,2025,1 Seasons,"Comedy, Drama",te,Sammelanam is a gripping Telugu web series tha...,3.236,0,0.00,sammelanam,netflix_tv_shows_detailed_up_to_2025
15996,277665,TV Show,Anne Shirley,NaN,"Honoka Inoue, Yasunori Matsumoto, Aya Nakamura...",Japan,2025-04-05,2025,1 Seasons,"Animation, Drama, Family",ja,It's depict Anne's growth from a girl to a wom...,3.558,0,0.00,anne shirley,netflix_tv_shows_detailed_up_to_2025
15997,284972,TV Show,Le onde del passato,Giulio Manfredonia,"Anna Valle, Giorgio Marchesi, Irene Ferri, Fau...",NaN,2025-02-19,2025,1 Seasons,Drama,it,NaN,2.913,1,100.00,le onde del passato,netflix_tv_shows_detailed_up_to_2025
15998,284613,TV Show,AI히치하이커,NaN,NaN,NaN,2025-02-16,2025,1 Seasons,NaN,ko,NaN,2.787,0,0.00,ai히치하이커,netflix_tv_shows_detailed_up_to_2025


### Netflix Movies and TV Shows

In [9]:
netflix_titles = pd.read_csv('datasets/Netflix Movies and TV Shows/netflix_titles.csv')
netflix_titles

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [10]:
netflix_titles['title_normalized'] = netflix_titles['title'].apply(normalize_title)
netflix_titles['source_dataset'] = 'netflix_titles'
netflix_titles['date_added'] = pd.to_datetime(netflix_titles['date_added'], format='%B %d, %Y', errors='coerce')
netflix_titles.rename(columns={'show_id' : 'source_row_id'}, inplace=True)
netflix_titles

,source_row_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,title_normalized,source_dataset
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",dick johnson is dead,netflix_titles
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",blood and water,netflix_titles
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,ganglands,netflix_titles
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",jailbirds new orleans,netflix_titles
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,kota factory,netflix_titles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",zodiac,netflix_titles
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01,2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...",zombie dumb,netflix_titles
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,zombieland,netflix_titles
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",zoom,netflix_titles


### Netflix

In [11]:
netflix = pd.read_csv('datasets/Netflix/NetFlix.csv')
netflix

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,14-Aug-20,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,15-Dec-17,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,5-Jan-19,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,1-Mar-16,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...
4,s1001,TV Show,Blue Planet II,NaN,David Attenborough,United Kingdom,3-Dec-18,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...
...,...,...,...,...,...,...,...,...,...,...,...,...
7782,s995,TV Show,Blown Away,NaN,NaN,Canada,12-Jul-19,2019,TV-14,1,"International TV Shows, Reality TV",Ten master artists turn up the heat in glassbl...
7783,s996,TV Show,Blue Exorcist,NaN,"Nobuhiko Okamoto, Jun Fukuyama, Kana Hanazawa,...",Japan,1-Sep-20,2017,TV-MA,2,"Anime Series, International TV Shows",Determined to throw off the curse of being Sat...
7784,s997,Movie,Blue Is the Warmest Color,Abdellatif Kechiche,"Léa Seydoux, Adèle Exarchopoulos, Salim Kechio...","France, Belgium, Spain",26-Aug-16,2013,NC-17,180,"Dramas, Independent Movies, International Movies","Determined to fall in love, 15-year-old Adele ..."
7785,s998,Movie,Blue Jasmine,Woody Allen,"Cate Blanchett, Sally Hawkins, Alec Baldwin, L...",United States,8-Mar-19,2013,PG-13,98,"Comedies, Dramas, Independent Movies",The high life leads to high anxiety for a fash...


In [12]:
netflix['title_normalized'] = netflix['title'].apply(normalize_title)
netflix['date_added'] = pd.to_datetime(netflix['date_added'], errors='coerce')
netflix.rename(columns={'show_id' : 'source_row_id'}, inplace=True)
netflix

/var/folders/rb/jjbql8_n2svfm4_nn63dwtt40000gn/T/ipykernel_54286/2599876928.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  netflix['date_added'] = pd.to_datetime(netflix['date_added'], errors='coerce')


,source_row_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description,title_normalized
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,3%
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,1920
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,3 heroines
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...,blue mountain state the rise of thadland
4,s1001,TV Show,Blue Planet II,NaN,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,blue planet ii
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7782,s995,TV Show,Blown Away,NaN,NaN,Canada,2019-07-12,2019,TV-14,1,"International TV Shows, Reality TV",Ten master artists turn up the heat in glassbl...,blown away
7783,s996,TV Show,Blue Exorcist,NaN,"Nobuhiko Okamoto, Jun Fukuyama, Kana Hanazawa,...",Japan,2020-09-01,2017,TV-MA,2,"Anime Series, International TV Shows",Determined to throw off the curse of being Sat...,blue exorcist
7784,s997,Movie,Blue Is the Warmest Color,Abdellatif Kechiche,"Léa Seydoux, Adèle Exarchopoulos, Salim Kechio...","France, Belgium, Spain",2016-08-26,2013,NC-17,180,"Dramas, Independent Movies, International Movies","Determined to fall in love, 15-year-old Adele ...",blue is the warmest color
7785,s998,Movie,Blue Jasmine,Woody Allen,"Cate Blanchett, Sally Hawkins, Alec Baldwin, L...",United States,2019-03-08,2013,PG-13,98,"Comedies, Dramas, Independent Movies",The high life leads to high anxiety for a fash...,blue jasmine


### Arpita-deb-netflix-movies-and-tv-shows

In [13]:
titles_cleaned = pd.read_csv('datasets/Arpita-deb-netflix-movies-and-tv-shows/titles_cleaned.csv')
credits_cleaned = pd.read_csv('datasets/Arpita-deb-netflix-movies-and-tv-shows/credits_cleaned.csv')
country = pd.read_csv('datasets/Arpita-deb-netflix-movies-and-tv-shows/country.csv')
titles_cleaned.head()

,id,title,type,release_year,age_certification,runtime,seasons,imdb_score,imdb_votes,country,genre
0,tm257629,Followfriday,Movie,2016,Others,90,0,2.7,382,US,Thriller
1,tm286268,Rucker50,Movie,2016,Others,56,0,5.2,117,US,Documentation
2,tm320206,Realityhigh,Movie,2017,Others,99,0,5.2,6283,US,"Comedy, Drama, Romance"
3,tm371188,Friendbutmarried,Movie,2018,Others,102,0,6.9,695,ID,"Romance, Comedy, Drama"
4,tm820190,Alive,Movie,2020,Others,98,0,6.3,37199,KR,"Thriller, Horror, Action, Drama"


In [14]:
credits_cleaned.head()

,person_id,id,name,role,character
0,509010,tm14491,Jackie Curtis,Actor,Self
1,271137,ts223454,Jason Young,Actor,NaN
2,105220,tm94651,Zeenat Aman,Actor,Sheetal Sahni
3,124506,ts258415,Roni Ezra,Director,Director
4,70582,tm156453,Paul Mooney,Actor,Himself


In [15]:
country.head()

,id,country_code,country_name
0,1,AE,United Arab Emirates
1,2,AL,Albania
2,3,AO,Angola
3,4,AR,Argentina
4,5,AT,Austria


In [16]:
country_map = {c : name for c, name in zip(country['country_code'].to_list(), country['country_name'].to_list())}

def to_country_name(country_code):
    if pd.isna(country_code):
        return pd.NA
    cntrs = list(map(str.strip, country_code.split(',')))
    mapped = [country_map[c] for c in cntrs if c in country_map]
    return ', '.join(mapped)

titles_cleaned['country'] = titles_cleaned['country'].apply(to_country_name)

In [17]:
cast = {title_id : credits_cleaned[(credits_cleaned['id'] == title_id) & (credits_cleaned['role'] == 'Actor')]['name'].to_list() for title_id in titles_cleaned['id'].to_list()}
titles_cleaned['cast'] = titles_cleaned['id'].apply(lambda x: ', '.join(cast[x]) if cast[x] else pd.NA)

In [18]:
director = {title_id : credits_cleaned[(credits_cleaned['id'] == title_id) & (credits_cleaned['role'] == 'Director')]['name'].to_list() for title_id in titles_cleaned['id'].to_list()}
titles_cleaned['director'] = titles_cleaned['id'].apply(lambda x: ', '.join(director[x]) if director[x] else pd.NA)

In [19]:
titles_cleaned['title_normalized'] = titles_cleaned['title'].apply(normalize_title)
titles_cleaned['type'] = titles_cleaned['type'].apply(lambda x: 'TV Show' if x == 'Show' else x)

In [20]:
def get_duration(id):
    if titles_cleaned[titles_cleaned['id'] == id]['type'].values == 'Movie':
        return str(titles_cleaned[titles_cleaned['id'] == id]['runtime'].values[0]) + ' min'
    return str(titles_cleaned[titles_cleaned['id'] == id]['seasons'].values[0]) + (' Season' if titles_cleaned[titles_cleaned['id'] == id]['seasons'].values[0] == 1 else ' Seasons')
titles_cleaned['duration'] = titles_cleaned['id'].apply(get_duration)

In [21]:
titles_cleaned.rename(
    columns={
        'id' : 'source_row_id',
        'age_certification' : 'rating',
        'imdb_score' : 'user_rating_score',
        'imdb_votes' : 'user_rating_size'
    }, inplace=True
)
titles_cleaned['user_rating_score'] = titles_cleaned['user_rating_score'] * 10
titles_cleaned.drop(['seasons', 'runtime'], inplace=True, axis=1)

---

Соберем все в одном месте

In [ ]:
dataset = {}

for _set in [netflix_movies_detailed_up_to_2025, netflix_tv_shows_detailed_up_to_2025, titles_cleaned, NetflixShows_original, netflix_titles, netflix]:
    columns = _set.columns
    for id, row in _set.iterrows():
        key = (row['title'], row['release_year'], row['type'])
        if key in dataset:
            for column in columns:
                if pd.isna(dataset[key].get(column, pd.NA)):
                    dataset[key][column] = row[column]
        else:
            dataset[key] = {}
            for column in columns:
                dataset[key][column] = row[column]

In [23]:
ds = pd.DataFrame({i : v  for (i, (k, v)) in enumerate(dataset.items(), start=1)}).T

In [24]:
ds = ds.drop(['source_row_id', 'source_dataset'], axis=1)
ds['title_id'] = [f't{i}' for i in range(1, len(ds) + 1)]
ds

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,popularity,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id
1,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,"Comedy, Adventure, Fantasy, Animation, Family",en,...,203.893,7449,63.8,165000000,752600867,shrek forever after,PG,"Fantasy, Comedy, Family, Romance, Animation",NaN,t1
2,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,"Action, Science Fiction, Adventure",en,...,156.242,37119,83.69,160000000,839030630,inception,PG-13,"Scifi, Music, Thriller, Action","Action & Adventure, Sci-Fi & Fantasy, Thrillers",t2
3,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,"Adventure, Fantasy",en,...,121.191,19327,77.44,250000000,954305868,harry potter and the deathly hallows part 1,NaN,NaN,NaN,t3
4,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,"Animation, Family, Adventure",en,...,111.762,11638,76.0,260000000,592461732,tangled,NaN,NaN,NaN,t4
5,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,"Fantasy, Adventure, Animation, Family",en,...,110.044,13259,78.0,165000000,494879471,how to train your dragon,NaN,NaN,NaN,t5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41467,TV Show,Wynonna Earp,NaN,"Melanie Scrofano, Shamier Anderson, Tim Rozon,...","Canada, United States",2019-07-16 00:00:00,2018,3,"International TV Shows, TV Action & Adventure,...",NaN,...,NaN,NaN,NaN,NaN,NaN,wynonna earp,TV-14,NaN,NaN,t41467
41468,TV Show,Zig & Sharko,NaN,NaN,France,2017-12-01 00:00:00,2016,1,"Kids' TV, TV Comedies",NaN,...,NaN,NaN,NaN,NaN,NaN,zig and sharko,TV-Y7,NaN,NaN,t41468
41469,TV Show,Ben & Holly's Little Kingdom,NaN,"Preston Nyman, Sian Taylor, Ian Puleston-Davie...",United Kingdom,2020-03-15 00:00:00,2009,1,Kids' TV,NaN,...,NaN,NaN,NaN,NaN,NaN,ben and holly's little kingdom,TV-Y,NaN,NaN,t41469
41470,TV Show,Black Lightning,NaN,"Cress Williams, James Remar, Marvin 'Krondon' ...",United States,2020-03-17 00:00:00,2019,3,"TV Action & Adventure, TV Dramas, TV Sci-Fi & ...",NaN,...,NaN,NaN,NaN,NaN,NaN,black lightning,TV-14,NaN,NaN,t41470


In [25]:
ds.to_csv('datasets/netflix_merged.tsv', index=False, sep='\t')